## Import Libraries

In [1]:
from pathlib import Path
import sys
import importlib
import itertools
import warnings
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

try:
    import tensorflow as tf
    from tensorflow.keras import Sequential
    from tensorflow.keras.layers import Dense, Dropout, Input
    from tensorflow.keras.callbacks import EarlyStopping
    from tensorflow.keras.optimizers import Adam

    tf.keras.utils.set_random_seed(RANDOM_STATE)
    print("TensorFlow version:", tf.__version__)
except ImportError as exc:
    raise ImportError(
        "TensorFlow is required for the Multilayer ANN. "
        "Install it with `pip install tensorflow`, restart the kernel, "
        "and run the notebook again."
    ) from exc

TensorFlow version: 2.21.0


## Locate the project and reuse the updated preprocessing pipeline

In [2]:
current_dir = Path.cwd().resolve()

candidate_roots = [
    current_dir,
    current_dir.parent,
    current_dir.parent.parent,
]

project_root = next(
    (
        path
        for path in candidate_roots
        if (
            path / "data" / "raw" /
            "GlobalLandTemperaturesByCountry.csv"
        ).exists()
        and (path / "code" / "preprocess.py").exists()
    ),
    None
)

if project_root is None:
    raise FileNotFoundError(
        "Cannot find the project root. Expected both:\n"
        "  data/raw/GlobalLandTemperaturesByCountry.csv\n"
        "  code/preprocess.py\n"
        "Place this notebook inside the same project used by "
        "01_data_preprocessing.ipynb."
    )

sys.path.insert(0, str(project_root / "code"))

import preprocess as pp
importlib.reload(pp)

print("Project root:", project_root)

Project root: C:\Users\dlihi\Documents\PRT565 Machine Learning\Assessment 3\PRT565-Machine-Learning


## Run the updated proprocessing pipeline

In [3]:
results = pp.run_pipeline(project_root)

train_df = results["train"].copy()
test_df = results["test"].copy()

print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)

print("\nTraining reference years:")
print(sorted(train_df["reference_year"].unique()))

print("\nTesting reference years:")
print(sorted(test_df["reference_year"].unique()))

print("\nTraining class distribution:")
display(
    train_df["target_exposure_class"]
    .value_counts()
    .reindex(["Low", "Medium", "High"])
)

print("\nTesting class distribution:")
display(
    test_df["target_exposure_class"]
    .value_counts()
    .reindex(["Low", "Medium", "High"])
)

display(train_df.head())
display(test_df.head())

Preprocessing completed successfully.
Model-ready files saved to: C:\Users\dlihi\Documents\PRT565 Machine Learning\Assessment 3\PRT565-Machine-Learning\data\processed
Training shape: (663, 13)
Testing shape: (221, 13)

Training reference years:
[np.int64(1970), np.int64(1980), np.int64(1990)]

Testing reference years:
[np.int64(2000)]

Training class distribution:


target_exposure_class
Low       221
Medium    221
High      221
Name: count, dtype: int64


Testing class distribution:


target_exposure_class
Low       62
Medium    93
High      66
Name: count, dtype: int64

,country,cca3,reference_year,population_at_reference,density_at_reference_per_km2,population_growth_prior_decade_pct,prior_decade_mean_temp_c,prior_decade_warming_c_per_decade,prior_decade_detrended_volatility_c,prior_decade_mean_temp_uncertainty_c,area_km2,continent,target_exposure_class
0,Aruba,ABW,1970,59106,328.366667,NaN,28.199408,0.107460,0.189841,0.358767,180,North America,High
1,Aruba,ABW,1980,62267,345.927778,5.348019,28.167542,0.197955,0.287829,0.327217,180,North America,Low
2,Aruba,ABW,1990,65712,365.066667,5.532626,28.342575,-0.270318,0.302661,0.324258,180,North America,High
4,Afghanistan,AFG,1970,10752971,16.486471,NaN,13.961283,-0.322576,0.494226,0.411983,652230,Asia,Low
5,Afghanistan,AFG,1980,12486631,19.144521,16.122614,14.035983,0.108848,0.746613,0.412525,652230,Asia,Low


,country,cca3,reference_year,population_at_reference,density_at_reference_per_km2,population_growth_prior_decade_pct,prior_decade_mean_temp_c,prior_decade_warming_c_per_decade,prior_decade_detrended_volatility_c,prior_decade_mean_temp_uncertainty_c,area_km2,continent,target_exposure_class
3,Aruba,ABW,2000,89101,495.005556,35.593195,28.531633,0.322152,0.224315,0.317167,180,North America,High
7,Afghanistan,AFG,2000,19542982,29.963329,82.733565,14.732458,0.799712,0.324100,0.451775,652230,Asia,Low
11,Angola,AGO,2000,16394062,13.149966,38.596362,22.477567,0.413253,0.294703,0.494600,1246700,Africa,Low
15,Anguilla,AIA,2000,11047,121.395604,32.840308,27.266525,0.514823,0.175331,0.282558,91,North America,High
19,Albania,ALB,2000,3182021,110.686691,-3.430736,13.149892,0.307763,0.498491,0.328900,28748,Europe,Medium


## Validate the temporal split and prevent leakage

In [4]:
assert train_df["reference_year"].max() < 2000
assert (test_df["reference_year"] == 2000).all()

forbidden_features = {
    "future_warming_c_per_decade",
    "future_density_per_km2",
    "outcome_population",
    "future_exposure_index",
}

assert forbidden_features.isdisjoint(train_df.columns)
assert forbidden_features.isdisjoint(test_df.columns)

assert not train_df.duplicated(
    ["cca3", "reference_year"]
).any()

assert not test_df.duplicated(
    ["cca3", "reference_year"]
).any()

print("Temporal split and leakage checks passed.")

Temporal split and leakage checks passed.


## Define the models predictors

In [5]:
NUMERIC_FEATURES = [
    "population_at_reference",
    "density_at_reference_per_km2",
    "population_growth_prior_decade_pct",
    "prior_decade_mean_temp_c",
    "prior_decade_warming_c_per_decade",
    "prior_decade_detrended_volatility_c",
    "prior_decade_mean_temp_uncertainty_c",
    "area_km2",
]

CATEGORICAL_FEATURES = [
    "continent",
]

TARGET = "target_exposure_class"

MODEL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

missing_columns = [
    col
    for col in MODEL_FEATURES + [TARGET]
    if col not in train_df.columns
]

if missing_columns:
    raise KeyError(
        f"Expected columns are missing from the updated preprocessing output: "
        f"{missing_columns}"
    )

print("Numerical predictors:")
print(NUMERIC_FEATURES)

print("\nCategorical predictors:")
print(CATEGORICAL_FEATURES)

Numerical predictors:
['population_at_reference', 'density_at_reference_per_km2', 'population_growth_prior_decade_pct', 'prior_decade_mean_temp_c', 'prior_decade_warming_c_per_decade', 'prior_decade_detrended_volatility_c', 'prior_decade_mean_temp_uncertainty_c', 'area_km2']

Categorical predictors:
['continent']


## Encode the ordered target

In [6]:
CLASS_ORDER = ["Low", "Medium", "High"]

CLASS_TO_INT = {
    "Low": 0,
    "Medium": 1,
    "High": 2,
}

INT_TO_CLASS = {
    value: key
    for key, value in CLASS_TO_INT.items()
}

X_train_all = train_df[MODEL_FEATURES].copy()
X_test = test_df[MODEL_FEATURES].copy()

y_train_all = (
    train_df[TARGET]
    .astype(str)
    .map(CLASS_TO_INT)
    .to_numpy()
)

y_test = (
    test_df[TARGET]
    .astype(str)
    .map(CLASS_TO_INT)
    .to_numpy()
)

assert not pd.isna(y_train_all).any()
assert not pd.isna(y_test).any()

print("Class mapping:", CLASS_TO_INT)

Class mapping: {'Low': 0, 'Medium': 1, 'High': 2}


## Create an internal temporal validation split

In [7]:
dev_train_df = train_df[
    train_df["reference_year"].isin([1970, 1980])
].copy()

dev_val_df = train_df[
    train_df["reference_year"] == 1990
].copy()

X_dev_train = dev_train_df[MODEL_FEATURES].copy()
X_dev_val = dev_val_df[MODEL_FEATURES].copy()

y_dev_train = (
    dev_train_df[TARGET]
    .astype(str)
    .map(CLASS_TO_INT)
    .to_numpy()
)

y_dev_val = (
    dev_val_df[TARGET]
    .astype(str)
    .map(CLASS_TO_INT)
    .to_numpy()
)

print("Development training rows:", len(dev_train_df))
print("Temporal validation rows:", len(dev_val_df))
print("Final temporal test rows:", len(test_df))

assert set(dev_train_df["reference_year"]) == {1970, 1980}
assert set(dev_val_df["reference_year"]) == {1990}
assert set(test_df["reference_year"]) == {2000}

Development training rows: 442
Temporal validation rows: 221
Final temporal test rows: 221


## Build the shared preprocessing transformation

In [8]:
def make_preprocessor():
    try:
        one_hot = OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )
    except TypeError:
        one_hot = OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
        )

    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="median"),
            ),
            (
                "scaler",
                StandardScaler(),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                ),
            ),
            (
                "onehot",
                one_hot,
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                NUMERIC_FEATURES,
            ),
            (
                "categorical",
                categorical_pipeline,
                CATEGORICAL_FEATURES,
            ),
        ],
        remainder="drop",
    )


dev_preprocessor = make_preprocessor()

X_dev_train_processed = (
    dev_preprocessor.fit_transform(X_dev_train)
)

X_dev_val_processed = (
    dev_preprocessor.transform(X_dev_val)
)

print(
    "Development training matrix:",
    X_dev_train_processed.shape
)

print(
    "Validation matrix:",
    X_dev_val_processed.shape
)

Development training matrix: (442, 14)
Validation matrix: (221, 14)


## Helper functions for models selections and evaluation

In [9]:
def macro_f1(y_true, y_pred):
    return f1_score(
        y_true,
        y_pred,
        labels=[0, 1, 2],
        average="macro",
        zero_division=0,
    )


def evaluate_predictions(
    model_name,
    y_true,
    y_pred,
    y_probability=None,
):
    row = {
        "Model": model_name,
        "Accuracy": accuracy_score(
            y_true,
            y_pred,
        ),
        "Balanced Accuracy": balanced_accuracy_score(
            y_true,
            y_pred,
        ),
        "Macro Precision": precision_score(
            y_true,
            y_pred,
            labels=[0, 1, 2],
            average="macro",
            zero_division=0,
        ),
        "Macro Recall": recall_score(
            y_true,
            y_pred,
            labels=[0, 1, 2],
            average="macro",
            zero_division=0,
        ),
        "Macro F1": f1_score(
            y_true,
            y_pred,
            labels=[0, 1, 2],
            average="macro",
            zero_division=0,
        ),
    }

    if y_probability is not None:
        try:
            row["Macro ROC-AUC (OvR)"] = (
                roc_auc_score(
                    y_true,
                    y_probability,
                    labels=[0, 1, 2],
                    average="macro",
                    multi_class="ovr",
                )
            )
        except ValueError:
            row["Macro ROC-AUC (OvR)"] = np.nan
    else:
        row["Macro ROC-AUC (OvR)"] = np.nan

    return row

# Model 1 Decision Tree

In [10]:
decision_tree_candidates = []

for max_depth in [3, 5, 7, None]:
    for min_samples_leaf in [3, 5, 10, 20]:
        model = DecisionTreeClassifier(
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            random_state=RANDOM_STATE,
        )

        model.fit(
            X_dev_train_processed,
            y_dev_train,
        )

        val_pred = model.predict(
            X_dev_val_processed
        )

        decision_tree_candidates.append(
            {
                "max_depth": max_depth,
                "min_samples_leaf": min_samples_leaf,
                "Validation Macro F1": macro_f1(
                    y_dev_val,
                    val_pred,
                ),
            }
        )

dt_validation = (
    pd.DataFrame(decision_tree_candidates)
    .sort_values(
        "Validation Macro F1",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(dt_validation.head(10))

best_dt_depth = dt_validation.loc[
    0,
    "max_depth"
]

if pd.isna(best_dt_depth):
    best_dt_depth = None
else:
    best_dt_depth = int(best_dt_depth)

best_dt_leaf = int(
    dt_validation.loc[
        0,
        "min_samples_leaf"
    ]
)

print(
    "Selected Decision Tree settings:",
    {
        "max_depth": best_dt_depth,
        "min_samples_leaf": best_dt_leaf,
    }
)

,max_depth,min_samples_leaf,Validation Macro F1
0,7.0,5,0.591846
1,5.0,5,0.582632
2,7.0,3,0.579399
3,5.0,3,0.578500
4,NaN,5,0.554487
5,NaN,3,0.545462
6,5.0,10,0.544227
7,3.0,20,0.542360
8,NaN,20,0.527275
9,7.0,20,0.527275


Selected Decision Tree settings: {'max_depth': 7, 'min_samples_leaf': 5}


# Model 2 Random Forest

In [ ]:
random_forest_candidates = []

for max_depth in [6, 10, None]:
    for min_samples_leaf in [1, 2, 5]:
        for max_features in ["sqrt", 0.7]:
            model = RandomForestClassifier(
                n_estimators=400,
                max_depth=max_depth,
                min_samples_leaf=min_samples_leaf,
                max_features=max_features,
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )

            model.fit(
                X_dev_train_processed,
                y_dev_train,
            )

            val_pred = model.predict(
                X_dev_val_processed
            )

            random_forest_candidates.append(
                {
                    "max_depth": max_depth,
                    "min_samples_leaf": min_samples_leaf,
                    "max_features": max_features,
                    "Validation Macro F1": macro_f1(
                        y_dev_val,
                        val_pred,
                    ),
                }
            )

rf_validation = (
    pd.DataFrame(random_forest_candidates)
    .sort_values(
        "Validation Macro F1",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(rf_validation.head(10))

best_rf_depth = rf_validation.loc[
    0,
    "max_depth"
]

if pd.isna(best_rf_depth):
    best_rf_depth = None
else:
    best_rf_depth = int(best_rf_depth)

best_rf_leaf = int(
    rf_validation.loc[
        0,
        "min_samples_leaf"
    ]
)

best_rf_features = rf_validation.loc[
    0,
    "max_features"
]

print(
    "Selected Random Forest settings:",
    {
        "max_depth": best_rf_depth,
        "min_samples_leaf": best_rf_leaf,
        "max_features": best_rf_features,
    }
)